<a href="https://colab.research.google.com/github/raymondsum2002-wq/HongKongJockeyClub_HorseRacePrediction/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import sqlite3

st.set_page_config(page_title="HKJC v9.42 賽馬預測系統", layout="wide")

st.title("🏇 HKJC v9.42 賽馬預測與即時賠率分析系統")

# 1. 側邊欄：場次選擇與主持評分輸入
st.sidebar.header("🎯 賽事設定")
race_num = st.sidebar.number_input("選擇場次", min_value=1, max_value=11, value=1)

st.sidebar.subheader("🎙️ 臨場主持人狀態評分 (1-4分)")
use_presenter = st.sidebar.checkbox("啟用主持人臨場微調", value=False)
p_a_score = 3.0
p_b_score = 3.0
if use_presenter:
    p_a_score = st.sidebar.slider("主持 A 狀態", 1.0, 4.0, 3.0, 0.5)
    p_b_score = st.sidebar.slider("主持 B 狀態", 1.0, 4.0, 3.0, 0.5)

# 2. 數據載入與預測 (模擬 API 介面)
@st.cache_data
def load_race_data(r_num):
    # 實際運作時連接 sqlite3 或呼叫 predict_upcoming_racecard
    data = {
        '馬號': [1, 2, 3, 4, 5, 6, 7, 8],
        '馬名': ['金槍六福', '浪漫勇士', '將王', '加州星球', '飛輪閃耀', '美麗同享', '幸福笑容', '多巴先生'],
        'S1': [0.12, 0.15, 0.08, 0.10, 0.05, 0.07, 0.06, 0.04],
        'S2': [0.22, 0.25, 0.18, 0.20, 0.15, 0.12, 0.10, 0.08],
        'S3': [0.24, 0.22, 0.19, 0.18, 0.12, 0.15, 0.11, 0.09],
        'S4': [0.18, 0.19, 0.15, 0.12, 0.10, 0.08, 0.09, 0.05],
        'S5': [0.23, 0.21, 0.17, 0.19, 0.14, 0.11, 0.08, 0.07],
        '即時獨贏賠率': [3.5, 2.8, 8.5, 6.0, 15.0, 22.0, 35.0, 50.0],
        '快操變動': ['+2 (主帥親試)', '+1', '0', '-1', '+3 (練馬師親試)', '0', '-2', '0'],
        '上賽評語': ['走勢強勁，衝刺出色', '狀態大勇，輕鬆獲勝', '後追不及，形勢受阻', '前速快，末段微倦', '保持水準', '後勁尚可', '表現平平', '需要減分']
    }
    df = pd.DataFrame(data)

    # 計算 Model_Score
    opt_w = [0.1111, 0.2352, 0.2350, 0.1871, 0.2316]
    df['Model_Score'] = (opt_w[0]*df['S1'] + opt_w[1]*df['S2'] + opt_w[2]*df['S3'] +
                         opt_w[3]*df['S4'] + opt_w[4]*df['S5'])
    return df

df_race = load_race_data(race_num)

# 主持微調處置
if use_presenter:
    df_race['Model_Score'] += ((p_a_score + p_b_score)/2.0 - 3.0) * 0.02

# 3. 標註五大子模型最高分 Icon
icon_map = {'S1': '⏱️', 'S2': '⚡', 'S3': '🔥', 'S4': '🎯', 'S5': '💰'}
sub_cols = ['S1', 'S2', 'S3', 'S4', 'S5']

def assign_icons(row):
    icons = []
    for col in sub_cols:
        if row[col] == df_race[col].max():
            icons.append(icon_map[col])
    return " ".join(icons)

df_race['子模型最高分'] = df_race.apply(assign_icons, axis=1)

# 4. 計算理論賠率與 Value Score
exp_scores = np.exp(df_race['Model_Score'])
df_race['預估勝率'] = exp_scores / exp_scores.sum()
df_race['理論獨贏賠率'] = (0.825 / df_race['預估勝率']).round(1)
df_race['Value_Score'] = (df_race['即時獨贏賠率'] / df_race['理論獨贏賠率']).round(2)

# 5. UI 展示：排位表與動態排序
st.subheader(f"📋 第 {race_num} 場 排位與預測總覽")

sort_by = st.selectbox("排序方式", ['Model_Score (模型綜合得分)', 'Value_Score (值博率)', '即時獨贏賠率', 'S3 (近況狀態)'])
sort_col_map = {
    'Model_Score (模型綜合得分)': 'Model_Score',
    'Value_Score (值博率)': 'Value_Score',
    '即時獨贏賠率': '即時獨贏賠率',
    'S3 (近況狀態)': 'S3'
}

df_sorted = df_race.sort_values(by=sort_col_map[sort_by], ascending=(sort_by == '即時獨贏賠率')).reset_index(drop=True)

# 顯示表格
show_cols = ['馬號', '馬名', '子模型最高分', 'Model_Score', '理論獨贏賠率', '即時獨贏賠率', 'Value_Score', '快操變動', '上賽評語']
st.dataframe(
    df_sorted[show_cols].style.highlight_between(subset=['Value_Score'], left=1.25, right=10.0, color='#c6f6d5')
                       .highlight_between(subset=['Value_Score'], left=0.0, right=0.80, color='#fed7d7')
                       .format({'Model_Score': '{:.4f}', '預估勝率': '{:.1%}', 'Value_Score': '{:.2f}'}),
    use_container_width=True
)

# 6. 賠率走勢圖模組
st.subheader("📈 即時賠率走勢與落飛 Highlight")
time_slots = ['12:00', '18:00', '22:00', '08:00', '10:00 (賽前30m)', '10:20 (賽前10m)', '10:27 (賽前3m)']
odds_trend_data = {
    '時間': np.tile(time_slots, len(df_race)),
    '馬名': np.repeat(df_race['馬名'], len(time_slots)),
    '賠率': np.random.uniform(2.5, 40.0, len(time_slots) * len(df_race))
}
df_trend = pd.DataFrame(odds_trend_data)

fig = px.line(df_trend, x='時間', y='賠率', color='馬名', markers=True, title="WP 賠率變化曲線")
st.plotly_chart(fig, use_container_width=True)
